<a href="https://colab.research.google.com/github/552radhika-Dev/Final_Capstoneproj_Radhika/blob/main/part2/part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Part 2 — Supervised Machine Learning Model**

In [ ]:
#Loading the cleaned dataset file extracted from Part 1
import pandas as pd
import numpy as np
dataset_url = "https://raw.githubusercontent.com/552radhika-Dev/Final_Capstoneproj_Radhika/main/part2/cleaned_data.csv"
df = pd.read_csv(dataset_url)
print("Dataset loaded successfully.")
df.shape

In [ ]:
#Defining the Continuous Regression Target (y_reg) using selling_price column
y_reg = df['selling_price']
#Defining the Binary Classification Target (y_clf)
price_median = y_reg.median()
y_clf = (y_reg > price_median).astype(int)
print(f'Median selling price of cars:{price_median}')
#Defining the Feature Matrix (X) dropping the target variable to prevent data leakage
X = df.drop(columns=['selling_price', 'name'])
print(f'Shape of data post removing target variable columns:\n{X.shape}')
print(f'Shape of continuous numeric column(Selling Price):\n{y_reg.shape}')
print(f'Binarized selling price column:\n{y_clf}')

'name' column has thousands of unique values. It will create thousands of new columns, cluttering the feature matrix, consuming massive memory, and leading to overfitting. Hence dropping 'name' column along with 'Selling price'column

In [ ]:
df.head(5)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Re-initialize X to ensure it contains original categorical columns
X = df.drop(columns=['selling_price', 'name'])

#Label Encoding for Ordinal Columns
ordered_col = {
    'First Owner': 1,
    'Second Owner': 2,
    'Third Owner': 3,
    'Fourth & Above Owner': 4,
    'Test Drive Car': 0
}

#Mapping the owner column post label encoding to preserve ordinal weight
X['owner'] = X['owner'].map(ordered_col)
#One-Hot Encoding for unordered columns
unordered_cols = ['fuel', 'seller_type', 'transmission']
#dropping the first column for memory utilisation and its unnecessary to keep
drop_col = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')
#Fitting and transforming the unordered columns
encoded_unordered = drop_col.fit_transform(X[unordered_cols])
#Generating clean, descriptive column names for the new binary features
encoded_cols = drop_col.get_feature_names_out(unordered_cols)
df_encoded_unordered = pd.DataFrame(encoded_unordered, columns=encoded_cols, index=X.index)
#Drop original unordered columns and joining the new one-hot encoded dummy variables
X = X.drop(columns=unordered_cols).join(df_encoded_unordered)
print(f"Current Features Matrix X shape: {X.shape}")
print(f"Total columns post encoding:\n{X.dtypes}")

In [ ]:
X.head(5)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#Leak-Free Train-Test Split
#Splitting the feature matrix and both the regression and classification targets
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape[0]} samples")
print(f"Test Set: {X_test.shape[0]} samples\n")

#Initializing the standard scaler tool
scaler = StandardScaler()

scaler.fit(X_train)
# Transforming both sets using the rules learned strictly from X_train
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Converting arrays back to DataFrames to preserve readable structural columns
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

#Linear regression
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_reg_train)
#Predictions
y_pred_train_linear = linear_model.predict(X_train_scaled)
y_pred_test_linear = linear_model.predict(X_test_scaled)
#Metrics
mse_train_linear = mean_squared_error(y_reg_train, y_pred_train_linear)
mse_test_linear = mean_squared_error(y_reg_test, y_pred_test_linear)
r2_train_linear = r2_score(y_reg_train, y_pred_train_linear)
r2_test_linear = r2_score(y_reg_test, y_pred_test_linear)


#Feature Importance / Coefficients Analysis
coefficients_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': linear_model.coef_,
    'Abs_Coefficient': np.abs(linear_model.coef_)
}).sort_values(by='Abs_Coefficient', ascending=False)
print("Linear Regression Coefficients")
print(coefficients_df[['Feature', 'Coefficient']].to_string(index=False))
print("\nTop 3 Most Influential Features:")
print(coefficients_df.head(3)['Feature'].tolist())

#Ridge Regression Regularization ---
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_reg_train)
#Predictions
y_pred_train_ridge = ridge_model.predict(X_train_scaled)
y_pred_test_ridge = ridge_model.predict(X_test_scaled)
#Metrics
mse_train_ridge = mean_squared_error(y_reg_train, y_pred_train_ridge)
mse_test_ridge = mean_squared_error(y_reg_test, y_pred_test_ridge)
r2_train_ridge = r2_score(y_reg_train, y_pred_train_ridge)
r2_test_ridge = r2_score(y_reg_test, y_pred_test_ridge)

print("=== Linear Regression (OLS) Performance ===")
print(f"Train MSE: {mse_train_linear:,.2f} | Test MSE: {mse_test_linear:,.2f}")
print(f"Train R²:  {r2_train_linear:.4f}  | Test R²:  {r2_test_linear:.4f}\n")

print("=== Ridge Regression Performance ===")
print(f"Train MSE: {mse_train_ridge:,.2f} | Test MSE: {mse_test_ridge:,.2f}")
print(f"Train R²:  {r2_train_ridge:.4f}  | Test R²:  {r2_test_ridge:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

#Checking Class Imbalance
print(y_clf_train.value_counts(normalize=True))
print(y_clf_train.value_counts())
print("-" * 50)
#Training Logistic Regression
clf_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_model.fit(X_train_scaled, y_clf_train)
#Predictions
y_pred_clf = clf_model.predict(X_test_scaled)
y_pred_prob = clf_model.predict_proba(X_test_scaled)[:, 1]
#Metrics Evaluation
print("=== Confusion Matrix ===")
print(confusion_matrix(y_clf_test, y_pred_clf))
print("\n=== Classification Report ===")
print(classification_report(y_clf_test, y_pred_clf))
#ROC Curve & AUC Calculation
fpr, tpr, thresholds = roc_curve(y_clf_test, y_pred_prob)
auc_value = roc_auc_score(y_clf_test, y_pred_prob)
print(f"ROC AUC Score: {auc_value:.4f}")
#Plotting the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Logistic Regression (AUC = {auc_value:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')
#Graph Customization
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
#Annotate AUC value on the plot canvas
plt.text(0.6, 0.4, f'AUC = {auc_value:.4f}', fontsize=12,
         bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.3))

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
#Getting prediction probabilities for the positive class (Class 1)
y_prob_test = clf_model.predict_proba(X_test_scaled)[:, 1]
#Defining thresholds from 0.30 to 0.70 with steps of 0.10
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
#List to collect rows for our table
target_data = []

for t in thresholds:
    y_pred_custom = (y_prob_test >= t).astype(int)
    #Calculating performance metrics
    prec = precision_score(y_clf_test, y_pred_custom, zero_division=0)
    rec = recall_score(y_clf_test, y_pred_custom, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_custom, zero_division=0)
    #Appending results formatted to 4 decimal places
    target_data.append({
        "Threshold": f"{t:.2f}",
        "Precision": f"{prec:.4f}",
        "Recall": f"{rec:.4f}",
        "F1": f"{f1:.4f}"
    })

#Converting to DataFrame and display as a structured table
sensitivity_df = pd.DataFrame(target_data)
print("=== Decision-Threshold Table ===")
print(sensitivity_df.to_string(index=False))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import pandas as pd
#Training Strongly Regularized Model(C=0.01)
clf_reg_model = LogisticRegression(C=0.01, max_iter=1000, class_weight='balanced', random_state=42)
clf_reg_model.fit(X_train_scaled, y_clf_train)
#Generating Predictions
y_pred_reg_clf = clf_reg_model.predict(X_test_scaled)
y_prob_reg_clf = clf_reg_model.predict_proba(X_test_scaled)[:, 1]
#Computing Metrics for the Regularized Model
prec_reg = precision_score(y_clf_test, y_pred_reg_clf)
rec_reg = recall_score(y_clf_test, y_pred_reg_clf)
auc_reg = roc_auc_score(y_clf_test, y_prob_reg_clf)

#Pulling Metrics from Baseline Model (C=1.0)
#Assuming 'y_pred_clf' and 'y_pred_prob' from your previous baseline model run
prec_base = precision_score(y_clf_test, y_pred_clf)
rec_base = recall_score(y_clf_test, y_pred_clf)
auc_base = roc_auc_score(y_clf_test, y_pred_prob)

#Generating Comparison Table
comparison_data = {
    "Model Hyperparameter": ["Baseline (C=1.0)", "Strong Regularization (C=0.01)"],
    "Precision": [f"{prec_base:.4f}", f"{prec_reg:.4f}"],
    "Recall": [f"{rec_base:.4f}", f"{rec_reg:.4f}"],
    "ROC AUC": [f"{auc_base:.4f}", f"{auc_reg:.4f}"]
}

comparison_df = pd.DataFrame(comparison_data)
print("=== Logistic Regression Regularization Comparison ===")
print(comparison_df.to_string(index=False))

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
# y_prob_base -> probabilities from the C=1.0 baseline model
# y_prob_reg  -> probabilities from the C=0.01 regularized model
y_prob_base = clf_model.predict_proba(X_test_scaled)[:, 1]
y_prob_reg = clf_reg_model.predict_proba(X_test_scaled)[:, 1]
#Converting test targets to a numpy array for clean row indexing
y_test_arr = np.array(y_clf_test)
n_iterations = 500
n_samples = len(y_test_arr)
#Listing to store the AUC difference for each bootstrap sample
auc_differences = []
#Set random seed for reproducibility
np.random.seed(42)

for _ in range(n_iterations):
    #Sample row indices with replacement
    indices = np.random.choice(n_samples, size=n_samples, replace=True)

    #Indexing targets and prediction probabilities for this bootstrap sample
    y_sample = y_test_arr[indices]
    prob_base_sample = y_prob_base[indices]
    prob_reg_sample = y_prob_reg[indices]

    #Skipping iteration if bootstrap sample contains only one class (edge case protection)
    if len(np.unique(y_sample)) < 2:
        continue

    #Computing individual AUC scores
    auc_base = roc_auc_score(y_sample, prob_base_sample)
    auc_reg = roc_auc_score(y_sample, prob_reg_sample)

    #Calculating difference (C=1.0 minus C=0.01)
    diff = auc_base - auc_reg
    auc_differences.append(diff)

#Computing mean and 95% confidence intervals (2.5th and 97.5th percentiles)
mean_diff = np.mean(auc_differences)
lower_ci = np.percentile(auc_differences, 2.5)
upper_ci = np.percentile(auc_differences, 97.5)

print("=== Bootstrap Confidence Interval Results ===")
print(f"Mean AUC Difference (C=1.0 - C=0.01): {mean_diff:.6f}")
print(f"95% Confidence Interval boundaries: [{lower_ci:.6f}, {upper_ci:.6f}]")